# Projeto 01 - Xadrez

### Importando bibliotecas

In [1]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm
import math

### Inicializando a cena

In [2]:
# Cria a janela
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.TRUE);
window = glfw.create_window(800, 800, "Xadrez", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)

# Shaders em 3D
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

# Aloca
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

# Associa
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

# Compila
glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")

glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

# Linkagem
glAttachShader(program, vertex)
glAttachShader(program, fragment)

glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')

glUseProgram(program)

loc = glGetAttribLocation(program, "position")
loc_color = glGetUniformLocation(program, "color")
loc_transformation = glGetUniformLocation(program, "mat_transformation")

### Objetos

In [3]:
vertices = []

def cubo(c, lx, ly, lz):
    global vertices

    lx /= 2
    ly /= 2
    lz /= 2
    
    vertices += [
        # Bottom
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),
        (c[0]-lx, c[1]-ly, c[2]+lz),

        # Top
        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),

        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),

        # Front
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]-ly, c[2]-lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),

        # Back
        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),

        # Left
        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]-ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),

        (c[0]-lx, c[1]-ly, c[2]-lz),
        (c[0]-lx, c[1]+ly, c[2]+lz),
        (c[0]-lx, c[1]+ly, c[2]-lz),

        # Right
        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),
        (c[0]+lx, c[1]-ly, c[2]+lz),

        (c[0]+lx, c[1]-ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]-lz),
        (c[0]+lx, c[1]+ly, c[2]+lz),
    ]

class Objeto:
    def __init__(self, id, inicio, quant, cor=[0,0,0]):
        self.id = id
        self.inicio = inicio
        self.quant = quant
        self.cor = cor

objetos = []

In [4]:
COR = {
    "fundo": [0.784, 0.949, 0.933],
    "tabuleiro": {
        "base": [0.588, 0.392, 0.212],
        "brancas": [0.868, 0.879, 0.77],
        "pretas": [0.463, 0.584, 0.341],
    },
    "peça": {
        "brancas": [1, 1, 1],
        "pretas": [0.1, 0.1, 0.1],
    }
}

# Tabuleiro (1/5)
grid = np.zeros((8, 8, 3), dtype=np.float32)
ds = 0.4/8
for i in range(8):
    for j in range(8):
        x = -0.2 + ds/2 + i*ds
        z = -0.2 + ds/2 + j*ds
        grid[i,j] = [x, -0.15, z]

        inicio = len(vertices)
        cubo([x, -0.15, z], ds, 0.08, ds)
        quant = len(vertices)-inicio
        if (i + j) % 2 != 0:
            objetos.append(Objeto("tablueiro", inicio, quant, COR["tabuleiro"]["brancas"]))
        else:
            objetos.append(Objeto("tablueiro", inicio, quant, COR["tabuleiro"]["pretas"]))

inicio = len(vertices)
cubo([0,-0.16,0], 0.5, 0.08, 0.5)
quant = len(vertices)-inicio
objetos.append(Objeto("tablueiro", inicio, quant, COR["tabuleiro"]["base"]))

# Peças
# Peão branco (2/5)
inicio = len(vertices)
cubo(grid[1,3]+[0,0.0425,0], ds/2, 0.005, ds/2)
cubo(grid[1,3]+[0,0.054,0], ds/4, 0.018, ds/4)
cubo(grid[1,3]+[0,0.063+ds/7,0], ds/3.5, ds/3.5, ds/3.5)
quant = len(vertices)-inicio
objetos.append(Objeto("peao_branco", inicio, quant, COR["peça"]["brancas"]))

# Peão preto (2/5)
inicio = len(vertices)
cubo(grid[1,4]+[0,0.0425,0], ds/2, 0.005, ds/2)
cubo(grid[1,4]+[0,0.054,0], ds/4, 0.018, ds/4)
cubo(grid[1,4]+[0,0.063+ds/7,0], ds/3.5, ds/3.5, ds/3.5)
quant = len(vertices)-inicio
objetos.append(Objeto("peao_preto", inicio, quant, COR["peça"]["pretas"]))

# Bispo preto (3/5)
inicio = len(vertices)
cubo(grid[5,7]+[0,0.0425,0], ds/2, 0.005, ds/2)
cubo(grid[5,7]+[0,0.054,0], ds/4, 0.018, ds/4)
cubo(grid[5,7]+[0,0.063+ds/7,0], ds/3.5, ds/3.5, ds/3.5)
cubo(grid[5,7]+[0,0.0655+2*ds/7,0], ds/6, 0.005, ds/6)

# Torre preta (4/5) [a fazer]

# Rei branco (5/5) [a fazer]

# Rei preto (5/5) [a fazer]

quant = len(vertices)-inicio
objetos.append(Objeto("bispo_preto", inicio, quant, COR["peça"]["pretas"]))

vertices = np.array(vertices, dtype=np.float32)

### Configuração do buffer

In [5]:
buffer_VBO = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_DYNAMIC_DRAW)

glEnableVertexAttribArray(loc)

stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

### Loop principal

In [6]:
dt = 0.05

# Variáveis para incrementos relativos à translação
vt_x = vt_z = 0

# Variáveis para incrementos relativos à rotação
va_y = 0

# Variável para incrementos relativos à escala uniforme
vsc = 0

# COMANDOS:
#   WASD - Movimentação do peão branco
#   QE - Rotação do tabuleiro
#   ZX - Zoom do tabuleiro
def key_event(window,key,scancode,action,mods):
    global vt_x, vt_z
    global va_y
    global vsc

    if action == glfw.PRESS or action == glfw.REPEAT:
        if key == glfw.KEY_W: vt_z = 0.1*dt
        if key == glfw.KEY_S: vt_z = -0.1*dt
        if key == glfw.KEY_A: vt_x = -0.1*dt
        if key == glfw.KEY_D: vt_x = 0.1*dt
    
        if key == glfw.KEY_E: va_y = dt
        if key == glfw.KEY_Q: va_y = -dt
    
        if key == glfw.KEY_X: vsc = dt
        if key == glfw.KEY_Z: vsc = -dt

    elif action == glfw.RELEASE:
        if key in (glfw.KEY_A, glfw.KEY_D): vt_x = 0
        if key in (glfw.KEY_W, glfw.KEY_S): vt_z = 0
        if key in (glfw.KEY_Q, glfw.KEY_E): va_y = 0
        if key in (glfw.KEY_Z, glfw.KEY_X): vsc = 0


glfw.set_key_callback(window,key_event)

In [7]:
# Matrizes de transformação 3D
def multiplica_matriz(a,b):
    m_a = a.reshape(4,4)
    m_b = b.reshape(4,4)
    m_c = np.dot(m_a,m_b)
    c = m_c.reshape(1,16)
    return c

def translacao(tx, ty, tz):
    return np.array([
        1, 0, 0, tx,
        0, 1, 0, ty,
        0, 0, 1, tz,
        0, 0, 0,  1
    ], np.float32)

def rotacao(rx, ry, rz):
    c, s = np.cos(rx), np.sin(rx)
    X = np.array([
        1, 0,  0, 0,
        0, c, -s, 0,
        0, s,  c, 0,
        0, 0,  0, 1
    ], np.float32)

    c, s = np.cos(ry), np.sin(ry)
    Y = np.array([
         c, 0, s, 0,
         0, 1, 0, 0,
        -s, 0, c, 0,
         0, 0, 0, 1
    ], np.float32)

    c, s = np.cos(rz), np.sin(rz)
    Z = np.array([
        c, -s, 0, 0,
        s,  c, 0, 0,
        0,  0, 1, 0,
        0,  0, 0, 1
    ], np.float32)

    YX = multiplica_matriz(Y,X)
    return multiplica_matriz(Z,YX)

def escala(sx, sy, sz):
    return np.array([
        sx,  0,  0, 0,
         0, sy,  0, 0,
         0,  0, sz, 0,
         0,  0,  0, 1
    ], np.float32)

In [8]:
glfw.show_window(window)

glEnable(GL_DEPTH_TEST)

# Variáveis para translação
t_x = t_z = 0

# Variável para rotação
a_y = 0

# Variável para escala uniforme
sc = 0

# Centroide
cx, cy, cz = vertices.mean(axis=0)

while not glfw.window_should_close(window):
    #glPolygonMode(GL_FRONT_AND_BACK,GL_LINE)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(COR["fundo"][0], COR["fundo"][1], COR["fundo"][2], 1.0)
    

    t_x += vt_x; t_z += vt_z
    a_y += va_y
    if ((2+sc > 0.7 and 2+sc < 3.3) or sc*vsc <= 0): sc += vsc

    t_x = np.clip(grid[1,3][0]+t_x, -0.175, 0.175)-grid[1,3][0]
    t_z = np.clip(grid[1,3][2]+t_z, -0.175, 0.175)-grid[1,3][2]

    for obj in objetos:
        if obj.id == "peao_branco": mat_transformation = translacao(t_x-cx,-cy,t_z-cz)
        else: mat_transformation = translacao(-cx,-cy,-cz)
        mat_transformation = multiplica_matriz(rotacao(-np.pi/4,a_y,0), mat_transformation)
        mat_transformation = multiplica_matriz(escala(2+sc,2+sc,2+sc), mat_transformation)
        mat_transformation = multiplica_matriz(translacao(cx, cy, cz), mat_transformation)
        glUniformMatrix4fv(loc_transformation, 1, GL_TRUE, mat_transformation) 
        
        glUniform4f(loc_color, obj.cor[0], obj.cor[1], obj.cor[2], 1.0)
        glDrawArrays(GL_TRIANGLE_STRIP, obj.inicio, obj.quant)

    
    glfw.swap_buffers(window)
    glfw.poll_events()

glfw.terminate()